# Session 7: Natural Language Processing in Healthcare

## 1. Setup

In [ ]:
!pip install -q torch torchvision transformers datasets pillow matplotlib numpy pandas scikit-learn requests

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from io import BytesIO
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. RNN Language Model

In [ ]:
medical_text = """
Patient presents with fever and cough. History of diabetes mellitus type 2.
Physical examination reveals increased respiratory rate and decreased oxygen saturation.
Chest radiograph shows bilateral infiltrates consistent with pneumonia.
Laboratory results indicate elevated white blood cell count and C-reactive protein.
Treatment initiated with broad-spectrum antibiotics and supportive care.
Patient admitted to medical ward for monitoring and further management.
Follow-up chest radiograph scheduled in 48 hours to assess treatment response.
Blood cultures obtained prior to antibiotic administration.
Glucose monitoring intensified due to underlying diabetes mellitus.
Patient counseled regarding smoking cessation and pneumococcal vaccination.
Prognosis is favorable with appropriate antibiotic therapy and supportive measures.
Discharge planning initiated with emphasis on medication adherence.
Outpatient follow-up arranged with primary care physician in two weeks.
Patient educated about warning signs requiring immediate medical attention.
Family members informed about infection control measures and home care.
Medical record documentation completed with detailed clinical notes.
Laboratory studies include complete blood count and metabolic panel.
Imaging studies reviewed by radiology department with formal report pending.
Consultation with infectious disease specialist requested for complex cases.
Antibiotic stewardship principles applied to optimize treatment regimen.
Patient age sixty two years with no known drug allergies.
Vital signs on admission include temperature thirty eight point five degrees Celsius.
Heart rate one hundred and ten beats per minute indicating tachycardia.
Blood pressure one forty over ninety millimeters of mercury.
Oxygen saturation eighty nine percent on room air requiring supplemental oxygen.
Arterial blood gas analysis shows hypoxemia and mild respiratory alkalosis.
Sputum culture sent to microbiology laboratory for organism identification.
Urinary antigen test performed for Legionella pneumophila and Streptococcus pneumoniae.
Intravenous fluids administered to maintain adequate hydration and perfusion.
Incentive spirometry encouraged to prevent atelectasis and improve lung expansion.
Deep vein thrombosis prophylaxis initiated with low molecular weight heparin.
Nutritional assessment performed by dietitian given prolonged hospitalization.
Electrolytes monitored daily including sodium potassium and magnesium levels.
Renal function tests ordered to guide antibiotic dosing and prevent nephrotoxicity.
Liver function panel obtained as baseline prior to initiation of hepatotoxic agents.
Hemoglobin A1c measured to assess long term glycemic control in diabetic patient.
Insulin sliding scale protocol implemented for inpatient glucose management.
Endocrinology consulted to optimize diabetes management during acute illness.
Fever pattern documented with temperature spikes recorded every four hours.
Respiratory therapist involved in management of supplemental oxygen delivery.
Patient placed in semi recumbent position to reduce aspiration risk.
Chest physiotherapy performed to assist secretion clearance and airway patency.
Bedside echocardiography performed to rule out cardiac effusion and dysfunction.
Electrocardiogram obtained to assess for arrhythmia in context of tachycardia.
Troponin levels measured to exclude myocardial injury from sepsis related stress.
Procalcitonin assay used to guide antibiotic de escalation decisions.
Serial clinical assessments conducted to monitor response to therapeutic interventions.
Nursing staff instructed on isolation precautions for respiratory pathogens.
Personal protective equipment utilized by all healthcare providers during patient contact.
Patient reported improvement in dyspnea and reduction in fever on day three.
Repeat laboratory results demonstrate declining inflammatory markers.
White blood cell count trending downward toward normal reference range.
C-reactive protein reduced by fifty percent compared to admission values.
Antibiotic regimen narrowed based on culture sensitivity results.
Oral antibiotic therapy substituted for intravenous route following clinical improvement.
Pulmonology consulted to evaluate for underlying structural lung disease.
Spirometry scheduled after recovery to assess baseline pulmonary function.
Patient mobilized with assistance of physiotherapist to prevent deconditioning.
Activity tolerance gradually increased with supervised ambulation.
Social work evaluation performed to identify barriers to medication adherence.
Pharmacy team reviewed complete medication list for interactions and duplications.
Patient provided written discharge instructions in preferred language.
Medication reconciliation completed prior to discharge from hospital.
Prescriptions issued for oral antibiotics and supplemental vitamin D.
Patient instructed to complete full antibiotic course to prevent resistance.
Return precautions discussed including worsening dyspnea and high grade fever.
Patient encouraged to maintain adequate fluid intake and rest during recovery.
Influenza vaccination offered and administered prior to discharge.
Pneumococcal polysaccharide vaccine administered to reduce future pneumonia risk.
Primary care physician notified of hospitalization and provided discharge summary.
Specialist follow up appointments scheduled within two weeks of discharge.
Telehealth option offered for follow up consultation if in person visit is difficult.
Patient expressed understanding of discharge plan and follow up requirements.
Caregiver training provided regarding wound care and home oxygen if applicable.
Home health services arranged for patients requiring continued nursing support.
Palliative care team involved for patients with advanced or complex disease burden.
Goals of care discussion documented in the electronic medical record.
Advance directive and healthcare proxy information reviewed and updated.
Patient rights and responsibilities reviewed at time of admission.
Informed consent obtained for all invasive procedures and diagnostic tests.
Interdisciplinary team meeting conducted to coordinate comprehensive discharge planning.
Case manager coordinated post discharge services and durable medical equipment.
Follow up laboratory tests ordered to confirm resolution of infection markers.
Repeat chest imaging deferred pending clinical reassessment at outpatient visit.
Long term risk stratification performed given history of diabetes and pneumonia.
Patient referred to pulmonary rehabilitation program to improve exercise capacity.
Smoking cessation pharmacotherapy prescribed with behavioral counseling referral.
Alcohol use screened and brief intervention provided as part of wellness assessment.
Mental health screening conducted given prolonged illness and hospitalization stress.
Anxiety and depression screening scores documented in clinical assessment.
Patient connected with community resources for chronic disease self management.
Electronic health record updated with comprehensive inpatient clinical summary.
Quality metrics reviewed including time to antibiotics and length of stay.
Hospital acquired infection checklist completed prior to patient transfer.
Antimicrobial resistance patterns reviewed with clinical pharmacist.
Multidrug resistant organism screening performed per institutional protocol.
Contact precautions maintained until culture results confirmed susceptibility.
Infection control nurse consulted for guidance on isolation and decontamination.
Environmental services notified to perform terminal cleaning of patient room.
Microbiological findings communicated to public health department as required.
Documentation standards reviewed to ensure compliance with regulatory requirements.
"""

chars = sorted(list(set(medical_text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}
vocab_size = len(chars)

print(f'Vocabulary size: {vocab_size}')
print(f'Text length: {len(medical_text)} characters')

In [ ]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, hidden):
        x = self.embedding(x)
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden
    
    def init_hidden(self, batch_size):
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)
        return (h0, c0)

model = CharRNN(vocab_size, hidden_size=128, num_layers=2).to(device)
print(model)

In [ ]:
seq_length = 50
sequences = []
targets = []

for i in range(len(medical_text) - seq_length):
    seq = medical_text[i:i+seq_length]
    target = medical_text[i+1:i+seq_length+1]
    sequences.append([char_to_idx[ch] for ch in seq])
    targets.append([char_to_idx[ch] for ch in target])

X = torch.tensor(sequences, dtype=torch.long)
y = torch.tensor(targets, dtype=torch.long)

dataset = torch.utils.data.TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f'Number of sequences: {len(sequences)}')

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)

num_epochs = 50
losses = []

for epoch in range(num_epochs):
    epoch_loss = 0
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        hidden = model.init_hidden(batch_x.size(0))
        optimizer.zero_grad()
        
        output, hidden = model(batch_x, hidden)
        loss = criterion(output.view(-1, vocab_size), batch_y.view(-1))
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('RNN Language Model Training Loss')
plt.grid(True)
plt.show()

In [ ]:
def generate_text(model, start_str='Patient', length=200, temperature=0.8):
    model.eval()
    with torch.no_grad():
        chars_generated = [ch for ch in start_str]
        hidden = model.init_hidden(1)
        
        for ch in start_str[:-1]:
            x = torch.tensor([[char_to_idx[ch]]]).to(device)
            output, hidden = model(x, hidden)
        
        x = torch.tensor([[char_to_idx[start_str[-1]]]]).to(device)
        
        for _ in range(length):
            output, hidden = model(x, hidden)
            output_dist = output.squeeze().div(temperature).exp()
            top_char_idx = torch.multinomial(output_dist, 1)[0].item()
            
            if top_char_idx < len(idx_to_char):
                next_char = idx_to_char[top_char_idx]
                chars_generated.append(next_char)
                x = torch.tensor([[top_char_idx]]).to(device)
        
        return ''.join(chars_generated)

generated = generate_text(model, start_str='Patient', length=200, temperature=0.8)
print('Generated medical text:')
print(generated)

## 3. RNN with Attention

In [ ]:
symptom_diagnosis_pairs = [
    ('fever cough shortness of breath', 'pneumonia'),
    ('chest pain radiating to left arm', 'myocardial infarction'),
    ('polyuria polydipsia weight loss', 'diabetes mellitus'),
    ('tremor rigidity bradykinesia', 'parkinson disease'),
    ('headache photophobia neck stiffness', 'meningitis'),
    ('abdominal pain nausea vomiting', 'gastroenteritis'),
    ('joint pain morning stiffness swelling', 'rheumatoid arthritis'),
    ('wheezing dyspnea chest tightness', 'asthma'),
    ('confusion fever altered mental status', 'encephalitis'),
    ('hematuria flank pain dysuria', 'urinary tract infection'),
    ('rash fever lymphadenopathy', 'viral infection'),
    ('fatigue weight gain cold intolerance', 'hypothyroidism'),
    ('palpitations anxiety tremor', 'hyperthyroidism'),
    ('jaundice abdominal distension ascites', 'liver cirrhosis'),
    ('hemoptysis night sweats weight loss', 'tuberculosis'),
    ('crushing chest pain diaphoresis syncope', 'angina pectoris'),
    ('sudden severe headache vomiting photophobia', 'subarachnoid hemorrhage'),
    ('productive cough fever pleuritic pain', 'bronchitis'),
    ('polyphagia fatigue blurred vision', 'hyperglycemia'),
    ('unilateral leg swelling warmth redness', 'deep vein thrombosis'),
    ('episodic vertigo tinnitus hearing loss', 'meniere disease'),
    ('burning epigastric pain relieved by food', 'peptic ulcer'),
    ('pruritus jaundice pale stools dark urine', 'cholestasis'),
    ('muscle weakness fatigue ptosis diplopia', 'myasthenia gravis'),
    ('recurrent seizures loss of consciousness', 'epilepsy'),
    ('excessive thirst urination fruity breath', 'diabetic ketoacidosis'),
    ('fever right upper quadrant pain murphy sign', 'cholecystitis'),
    ('progressive dyspnea orthopnea leg edema', 'heart failure'),
    ('skin thickening joint pain raynaud phenomenon', 'scleroderma'),
    ('severe flank pain nausea hematuria', 'renal calculi'),
]

all_words = set()
for symptoms, diagnosis in symptom_diagnosis_pairs:
    all_words.update(symptoms.split())
    all_words.update(diagnosis.split())

word_to_idx = {word: i+2 for i, word in enumerate(sorted(all_words))}
word_to_idx['<PAD>'] = 0
word_to_idx['<SOS>'] = 1
idx_to_word = {i: word for word, i in word_to_idx.items()}
vocab_size_attn = len(word_to_idx)

print(f'Vocabulary size: {vocab_size_attn}')
print(f'Number of symptom-diagnosis pairs: {len(symptom_diagnosis_pairs)}')

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
    
    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell

class Attention(nn.Module):
    def __init__(self, hidden_size):
        super(Attention, self).__init__()
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)
    
    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.size(1)
        hidden_repeated = hidden.unsqueeze(1).repeat(1, seq_len, 1)
        energy = torch.tanh(self.attn(torch.cat([hidden_repeated, encoder_outputs], dim=2)))
        attention_weights = torch.softmax(self.v(energy).squeeze(2), dim=1)
        context = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, attention_weights

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.attention = Attention(hidden_size)
        self.lstm = nn.LSTM(embed_size + hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, hidden, cell, encoder_outputs):
        embedded = self.embedding(x)
        context, attn_weights = self.attention(hidden[-1], encoder_outputs)
        lstm_input = torch.cat([embedded, context.unsqueeze(1)], dim=2)
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden, cell, attn_weights

embed_size = 64
hidden_size = 128

encoder = Encoder(vocab_size_attn, embed_size, hidden_size).to(device)
decoder = Decoder(vocab_size_attn, embed_size, hidden_size).to(device)

print('Encoder:', encoder)
print('\nDecoder:', decoder)

In [ ]:
def prepare_sequence(text, word_to_idx, max_len=10):
    words = text.split()
    indices = [word_to_idx.get(word, 0) for word in words]
    if len(indices) < max_len:
        indices += [0] * (max_len - len(indices))
    return indices[:max_len]

X_train = []
y_train = []

for symptoms, diagnosis in symptom_diagnosis_pairs:
    X_train.append(prepare_sequence(symptoms, word_to_idx))
    y_train.append(prepare_sequence(diagnosis, word_to_idx))

X_train = torch.tensor(X_train, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)

print(f'Input shape: {X_train.shape}')
print(f'Target shape: {y_train.shape}')

In [ ]:
criterion_attn = nn.CrossEntropyLoss(ignore_index=0)
encoder_optimizer = optim.Adam(encoder.parameters(), lr=0.001)
decoder_optimizer = optim.Adam(decoder.parameters(), lr=0.001)

num_epochs_attn = 100
losses_attn = []

for epoch in range(num_epochs_attn):
    epoch_loss = 0
    
    for i in range(len(X_train)):
        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()
        
        input_seq = X_train[i:i+1].to(device)
        target_seq = y_train[i:i+1].to(device)
        
        encoder_outputs, hidden, cell = encoder(input_seq)
        
        decoder_input = torch.tensor([[word_to_idx['<SOS>']]]).to(device)
        loss = 0
        num_valid_steps = 0
        
        for t in range(target_seq.size(1)):
            # Stop once we reach padding — all remaining tokens are PAD,
            # which would cause CrossEntropyLoss(ignore_index=0) to return nan.
            if target_seq[:, t].item() == 0:
                break
            
            output, hidden, cell, _ = decoder(decoder_input, hidden, cell, encoder_outputs)
            loss += criterion_attn(output, target_seq[:, t])
            num_valid_steps += 1
            decoder_input = target_seq[:, t].unsqueeze(1)
        
        if num_valid_steps == 0:
            continue
        
        # Normalize by number of valid tokens so sequences of different lengths
        # contribute equally-scaled gradients.
        loss = loss / num_valid_steps
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        torch.nn.utils.clip_grad_norm_(decoder.parameters(), max_norm=1.0)
        encoder_optimizer.step()
        decoder_optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(X_train)
    losses_attn.append(avg_loss)
    
    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs_attn}], Loss: {avg_loss:.4f}')

plt.figure(figsize=(10, 5))
plt.plot(losses_attn)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Seq2Seq with Attention Training Loss')
plt.grid(True)
plt.show()

In [ ]:
def predict_diagnosis(symptoms_text, encoder, decoder, word_to_idx, idx_to_word, max_len=10):
    encoder.eval()
    decoder.eval()
    
    with torch.no_grad():
        input_seq = torch.tensor([prepare_sequence(symptoms_text, word_to_idx)]).to(device)
        encoder_outputs, hidden, cell = encoder(input_seq)
        
        decoder_input = torch.tensor([[word_to_idx['<SOS>']]]).to(device)
        predicted_words = []
        attention_weights_list = []
        
        for _ in range(max_len):
            output, hidden, cell, attn_weights = decoder(decoder_input, hidden, cell, encoder_outputs)
            predicted_idx = output.argmax(1).item()
            
            if predicted_idx == 0:
                break
            
            predicted_word = idx_to_word.get(predicted_idx, '<UNK>')
            predicted_words.append(predicted_word)
            attention_weights_list.append(attn_weights.cpu().numpy())
            
            decoder_input = torch.tensor([[predicted_idx]]).to(device)
        
        return ' '.join(predicted_words), attention_weights_list, symptoms_text.split()

test_symptoms = 'fever cough shortness of breath'
predicted_diagnosis, attention_weights, input_tokens = predict_diagnosis(test_symptoms, encoder, decoder, word_to_idx, idx_to_word)
print(f'Symptoms: {test_symptoms}')
print(f'Predicted diagnosis: {predicted_diagnosis}')

if attention_weights and len(attention_weights) > 0:
    plt.figure(figsize=(10, 6))
    attn_matrix = np.array(attention_weights).squeeze()
    
    if len(attn_matrix.shape) == 1:
        attn_matrix = attn_matrix.reshape(1, -1)
    
    plt.imshow(attn_matrix, cmap='viridis', aspect='auto')
    plt.colorbar(label='Attention Weight')
    
    output_tokens = predicted_diagnosis.split()
    plt.yticks(range(len(output_tokens)), output_tokens)
    plt.xticks(range(len(input_tokens)), input_tokens, rotation=45, ha='right')
    
    plt.xlabel('Input Tokens (Symptoms)')
    plt.ylabel('Output Tokens (Diagnosis)')
    plt.title('Attention Weights Visualization')
    plt.tight_layout()
    plt.show()

## 4. Transformer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

medical_texts = [
    ("Patient diagnosed with type 2 diabetes mellitus and hypertension. Prescribed metformin and lisinopril.", "diabetes"),
    ("History of myocardial infarction with subsequent coronary artery bypass grafting. Currently on aspirin and statins.", "cardiovascular"),
    ("Persistent cough and dyspnea. Chest X-ray reveals infiltrates. Diagnosed with community-acquired pneumonia.", "respiratory"),
    ("Tremor, rigidity, and bradykinesia present. Clinical features consistent with Parkinson's disease.", "neurological"),
    ("Elevated blood glucose levels and polyuria. HbA1c of 8.5%. Diagnosis of diabetes mellitus confirmed.", "diabetes"),
    ("Acute chest pain radiating to left arm. ECG shows ST elevation. Urgent cardiac catheterization performed.", "cardiovascular"),
    ("Wheezing and shortness of breath triggered by allergens. Diagnosis of bronchial asthma established.", "respiratory"),
    ("Progressive memory loss and confusion. MRI reveals cortical atrophy. Alzheimer's disease suspected.", "neurological"),
    ("Fasting glucose 145 mg/dL. Patient counseled on diet and exercise. Started on oral hypoglycemic agents.", "diabetes"),
    ("Hypertension and hyperlipidemia managed with lifestyle modifications and pharmacotherapy.", "cardiovascular"),
    ("Chronic obstructive pulmonary disease with emphysema. Pulmonary function tests show airflow limitation.", "respiratory"),
    ("Seizure disorder controlled with anticonvulsant medication. EEG monitoring shows abnormal activity.", "neurological"),
    ("Diabetic retinopathy detected on fundoscopic examination. Referred to ophthalmology for evaluation.", "diabetes"),
    ("Atrial fibrillation with rapid ventricular response. Anticoagulation therapy initiated to prevent stroke.", "cardiovascular"),
    ("Pulmonary embolism confirmed by CT angiography. Treatment with anticoagulants commenced immediately.", "respiratory"),
    ("Multiple sclerosis with relapsing-remitting course. Disease-modifying therapy with interferons started.", "neurological"),
    ("Patient with poorly controlled diabetes presenting with diabetic ketoacidosis. Insulin therapy initiated.", "diabetes"),
    ("Type 1 diabetes with history of hypoglycemic episodes. Continuous glucose monitoring recommended.", "diabetes"),
    ("Gestational diabetes diagnosed during pregnancy. Blood glucose monitoring and dietary changes advised.", "diabetes"),
    ("Diabetic neuropathy with tingling and numbness in feet. Gabapentin prescribed for neuropathic pain.", "diabetes"),
    ("Congestive heart failure with reduced ejection fraction. ACE inhibitor and beta blocker therapy started.", "cardiovascular"),
    ("Peripheral artery disease with claudication. Referred for vascular surgery evaluation.", "cardiovascular"),
    ("Aortic stenosis detected on echocardiogram. Patient scheduled for valve replacement surgery.", "cardiovascular"),
    ("Deep vein thrombosis in lower extremity. Anticoagulation with warfarin initiated.", "cardiovascular"),
    ("Tuberculosis confirmed by sputum culture. Multi-drug therapy initiated with isolation precautions.", "respiratory"),
    ("Interstitial lung disease with progressive dyspnea. High-resolution CT shows fibrotic changes.", "respiratory"),
    ("Sleep apnea diagnosed with polysomnography. Continuous positive airway pressure therapy recommended.", "respiratory"),
    ("Pleural effusion detected on chest imaging. Thoracentesis performed for diagnostic and therapeutic purposes.", "respiratory"),
    ("Migraine headaches with visual aura. Prophylactic therapy with beta blockers initiated.", "neurological"),
    ("Stroke presenting with left-sided hemiparesis. Thrombolytic therapy administered within therapeutic window.", "neurological"),
    ("Peripheral neuropathy with sensory deficits. Vitamin B12 deficiency identified as underlying cause.", "neurological"),
    ("Guillain-Barré syndrome with ascending paralysis. Intravenous immunoglobulin therapy started.", "neurological"),
]

label_map = {"diabetes": 0, "cardiovascular": 1, "respiratory": 2, "neurological": 3}
texts = [item[0] for item in medical_texts]
labels = [label_map[item[1]] for item in medical_texts]

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels
)

train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
val_dataset = Dataset.from_dict({"text": val_texts, "label": val_labels})

print(f'Training samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print(f'Number of classes: {len(label_map)}')

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_transformer = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=len(label_map)
).to(device)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print('Model and tokenizer loaded successfully')

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="no",
)

trainer = Trainer(
    model=model_transformer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(train_result)

In [ ]:
eval_result = trainer.evaluate()
print("\nEvaluation Results:")
for key, value in eval_result.items():
    print(f"{key}: {value:.4f}")

test_text = "Patient presents with elevated fasting glucose and increased thirst. Diagnosed with diabetes."
inputs = tokenizer(test_text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

model_transformer.eval()
with torch.no_grad():
    outputs = model_transformer(**inputs)
    predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_class = predictions.argmax().item()

reverse_label_map = {v: k for k, v in label_map.items()}
print(f"\nTest text: {test_text}")
print(f"Predicted class: {reverse_label_map[predicted_class]}")
print(f"Confidence scores: {predictions.cpu().numpy()[0]}")

## 5. Vision-Language Model

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model_vlm = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

print('Vision-Language Model loaded successfully')

In [ ]:
medical_image_urls = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/0/07/Chest_Xray_PA_3-8-2010.png/400px-Chest_Xray_PA_3-8-2010.png",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/d/d8/Blausen_0620_Lungs_NormalXRay.png/400px-Blausen_0620_Lungs_NormalXRay.png",
]

def download_image(url, timeout=10, retries=2):
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()
            img = Image.open(BytesIO(response.content)).convert('RGB')
            return img
        except Exception as e:
            if attempt < retries - 1:
                print(f"Attempt {attempt + 1} failed, retrying...")
                continue
            else:
                print(f"Error downloading image after {retries} attempts: {e}")
                return None

def generate_caption(image, model, processor, device):
    inputs = processor(image, return_tensors="pt").to(device)
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_length=50)
    caption = processor.decode(out[0], skip_special_tokens=True)
    return caption

def answer_question(image, question, model, processor, device):
    inputs = processor(image, question, return_tensors="pt").to(device)
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_length=50)
    answer = processor.decode(out[0], skip_special_tokens=True)
    return answer

print('Image processing functions defined')

In [ ]:
for idx, url in enumerate(medical_image_urls):
    print(f"\n{'='*60}")
    print(f"Processing Image {idx + 1}")
    print(f"{'='*60}")
    
    image = download_image(url)
    
    if image is not None:
        plt.figure(figsize=(8, 6))
        plt.imshow(image)
        plt.axis('off')
        plt.title(f'Medical Image {idx + 1}')
        plt.show()
        
        caption = generate_caption(image, model_vlm, processor, device)
        print(f"Generated Caption: {caption}")
        
        question = "What organ is shown in this image?"
        answer = answer_question(image, question, model_vlm, processor, device)
        print(f"Question: {question}")
        print(f"Answer: {answer}")
    else:
        print(f"Skipping image {idx + 1} due to download failure")

In [ ]:
print("\nNotebook execution completed successfully!")
print("\nSummary:")
print("1. RNN Language Model: Trained character-level RNN on medical text")
print("2. RNN with Attention: Implemented seq2seq model for symptom-to-diagnosis mapping")
print("3. Transformer: Fine-tuned DistilBERT for medical text classification")
print("4. Vision-Language Model: Used BLIP for medical image captioning and VQA")